**Case Study: Stemming vs. Lemmatization in Sentiment Analysis of Movie Reviews**

**Objective:**
To compare the effectiveness of Stemming (using Porter Stemmer) and Lemmatization (using WordNetLemmatizer) in preprocessing text data for sentiment classification, evaluating their impact on model accuracy.

---

### Step 1: Import and Load Data

In [1]:
import nltk
import random # To shuffle data samples.
import pandas as pd

from nltk.corpus import movie_reviews # Dataset containing labeled movie reviews.
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag # Assigns part-of-speech tags (needed for Lemmatization).
from nltk.corpus import wordnet
import re #For regex-based text cleaning.

In [2]:
# Following all download statements, once executed, should be commented, so
# it wont download everytime one execute program.

nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Downloads a pre-trained part-of-speech (POS) tagger for NLTK,
# enabling word tagging (e.g., noun, verb) to improve lemmatization accuracy.

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\skuch\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\movie_reviews.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\skuch\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\skuch\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\skuch\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\skuch\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\skuch\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
movie_reviews

<CategorizedPlaintextCorpusReader in 'C:\\Users\\skuch\\AppData\\Roaming\\nltk_data\\corpora\\movie_reviews'>

In [5]:
# Load and shuffle 100 positive and 100 negative reviews

docs =[(movie_reviews.raw(fileid), category)
       for category in movie_reviews.categories()
       for fileid in movie_reviews.fileids(category)]

In [4]:
# Step 1: Get categories
# categories = ['neg', 'pos']

# docs = []
# for category in categories:  # Step 2: Outer Loop (category = 'neg' $\rightarrow$ 'pos')
#       fileids = movie_reviews.fileids(category)  # Step 3: Get file IDs for category
#       for fileid in fileids:  # Step 4: Inner Loop (each file)
#           text = movie_reviews.raw(fileid)  # Step 5: Read review text
#           docs.append((text, category))  # Step 6-7: Store (text, label)

# For each category, retrieves a list of file IDs
# (e.g., ['neg/cv000_29416.txt', 'neg/cv001_19502.txt', ...]).

# [("This movie was terrible...", "neg"),
#  ("I loved this film!", "pos"),

In [6]:
random.shuffle(docs)
docs = docs[:200] # first 200 reviews
docs

[('what were they thinking ? \nnostalgia for the seventies is bad enough , but do we really need an eighties film ? \nrobbie hart ( adam sandler ) used to want to be a rock and roll star , but in 1985 he\'s singing at weddings and having a good time . \na romantic at heart , he loves weddings and is just about to get married to his high-school sweetie . \nwhen she leaves him waiting at the altar , his tune changes to " love stinks " . \nhe meets waitress julia ( drew barrymore ) who is engaged to a junk-bonds salesman and you know that they are going to get together . \nin fact you know everything that is going to happen during this movie . \nsandler is somewhat adequate in his leading man role , but there is no spark . \nbarrymore doesn\'t seem to be able to convey anything other than a pretty face with nothing behind it : beauty but no attitude . \nboth characters are just there . \nbit parts by steve buscemi and jon lovitz steal the show . \nthe eighties are shoved in our face . \nr

In [7]:
df = pd.DataFrame(docs, columns= ['review', 'label'])
df['label'] = df['label'].map({'pos': 1, 'neg':0})

In [8]:
df.head()

,review,label
0,what were they thinking ? \nnostalgia for the ...,0
1,sometimes a movie comes along that falls somew...,1
2,"while watching "" shallow grave , "" i found mys...",1
3,sometimes a stellar cast can compensate for a ...,0
4,the previews for the movie are pretty good . \...,0


In [9]:
df.tail()


,review,label
195,much ballyhoo has been made over this new vers...,1
196,if you don't think kevin kline in drag is funn...,0
197,it was once said that in order to truly enjoy ...,0
198,there was probably a good reason that the warn...,0
199,"in the finale of disney's "" mighty joe young ,...",0


In [10]:
df.columns

Index(['review', 'label'], dtype='str')

In [11]:
df['label'].value_counts()

label
1    101
0     99
Name: count, dtype: int64

# step 2: text cleaning

In [12]:
def clean_text(text):
    text = text.lower() # Lowercase all text
    
    text = re.sub(r"[^\w\s]", "", text) # Remove punctuation , re is module for regex (regular expression)
    
    return text

df["clean"] = df["review"].apply(clean_text)

In [13]:
#Removes punctuation using regex (^^\w\s keeps only words/whitespace).

In [14]:
df['clean'].head()

0    what were they thinking  \nnostalgia for the s...
1    sometimes a movie comes along that falls somew...
2    while watching  shallow grave   i found myself...
3    sometimes a stellar cast can compensate for a ...
4    the previews for the movie are pretty good  \n...
Name: clean, dtype: str

In [15]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    

In [16]:
stop_words = set(stopwords.words('english')) - {'not','no', 'never'}

# removing negative words from stop word, so it can be used for sentiment analysis.


In [17]:
#Stemming

stemmer = PorterStemmer()
def stem_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    stemmed = [stemmer.stem(w) for w in filtered]
    return " ".join(stemmed)

# stemming process to text

In [20]:
df['stemmed'] = df['clean'].apply(stem_pipeline)

In [21]:
df['stemmed'].head()

0    think nostalgia seventi bad enough realli need...
1    sometim movi come along fall somewhat askew re...
2    watch shallow grave found period notic themat ...
3    sometim stellar cast compens lot thing push ti...
4    preview movi pretti good show littl plot chara...
Name: stemmed, dtype: str

# lemmatization

In [22]:
lemmatizer = WordNetLemmatizer()
def lemma_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    pos_tags = pos_tag(filtered)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in pos_tags]
    return " ".join(lemmatized)

# defined function for lemmatizing process.

In [24]:
nltk.download('averaged_perceptron_tagger_eng')

df['lemmatized'] = df['clean'].apply(lemma_pipeline)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\skuch\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


In [25]:
df['lemmatized'].head()

0    think nostalgia seventy bad enough really need...
1    sometimes movie come along fall somewhat askew...
2    watch shallow grave find periodically noticing...
3    sometimes stellar cast compensate lot thing pu...
4    preview movie pretty good show little plot cha...
Name: lemmatized, dtype: str

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
# Converts text to numerical TF-IDF feature vectors for machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Prepare data
X_stem = df["stemmed"]
X_lemma = df["lemmatized"]
y = df["label"]


In [29]:
X_stem.head()

0    think nostalgia seventi bad enough realli need...
1    sometim movi come along fall somewhat askew re...
2    watch shallow grave found period notic themat ...
3    sometim stellar cast compens lot thing push ti...
4    preview movi pretti good show littl plot chara...
Name: stemmed, dtype: str

In [28]:
X_lemma.head()

0    think nostalgia seventy bad enough really need...
1    sometimes movie come along fall somewhat askew...
2    watch shallow grave find periodically noticing...
3    sometimes stellar cast compensate lot thing pu...
4    preview movie pretty good show little plot cha...
Name: lemmatized, dtype: str

In [30]:
y.head()

0    0
1    1
2    1
3    0
4    0
Name: label, dtype: int64

In [31]:
# stemming data split
X_train_s, X_test_s, y_train, y_test = train_test_split(X_stem, y, test_size=0.2, random_state=42)

# lemit data split
X_train_l, X_test_l, _, _ = train_test_split(X_lemma, y, test_size=0.2, random_state=42)

# y_train and y_test are same for both, hence here skipped by writing _, _

In [33]:
X_train_s.shape

(160,)

In [34]:
y_train.shape

(160,)

In [35]:
# TF-IDF Vectorization

vectorizer = TfidfVectorizer()
X_train_s_vec = vectorizer.fit_transform(X_train_s)
X_test_s_vec = vectorizer.transform(X_test_s)


X_train_l_vec = vectorizer.fit_transform(X_train_l)
X_test_l_vec = vectorizer.transform(X_test_l)

In [36]:
# Train and evaluate
model_s = LogisticRegression(max_iter=1000)

model_s.fit(X_train_s_vec, y_train)

y_pred_s = model_s.predict(X_test_s_vec)

acc_s = accuracy_score(y_test, y_pred_s)

print("Stemming Accuracy:", round(acc_s, 2))

Stemming Accuracy: 0.75


In [37]:
model_l = LogisticRegression(max_iter=1000)

model_l.fit(X_train_l_vec, y_train)

y_pred_l = model_l.predict(X_test_l_vec)

acc_l = accuracy_score(y_test, y_pred_l)

print("Lemmatization Accuracy:", round(acc_l, 2))

Lemmatization Accuracy: 0.75


In [38]:
# Key Observations

# Stemming vs. Lemmatization:
# Stemming is faster but less accurate (e.g., "flies" $\rightarrow$ "fli").
# Lemmatization preserves meaning but requires POS tagging.

# Model Performance:
# Both methods achieved 75% accuracy in this case.
# Lemmatization may outperform in tasks needing semantic precision (e.g., "better" $\rightarrow$ "good").